In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.append("..")

from common import (
    LANGS,
    RUN_DIR,
    RES_DIR,
    load_benchmark,
    load_scores,
    record,
    paired_bootstrap_diff,
)

print("Languages:", LANGS)
print("Number of languages:", len(LANGS))

Languages: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Number of languages: 13


In [2]:
bench = {}

for lang in LANGS:
    bench[lang] = load_benchmark(lang)

for lang in LANGS:
    q, c, gold = bench[lang]
    print(
        f"{lang}: "
        f"queries={len(q)}, "
        f"docs={len(c)}, "
        f"gold={len(gold)}"
    )

as: queries=1000, docs=1000, gold=1000
bn: queries=1000, docs=1000, gold=1000
gu: queries=999, docs=999, gold=999
hi: queries=1000, docs=1000, gold=1000
kn: queries=1000, docs=1000, gold=1000
ml: queries=1000, docs=1000, gold=1000
mr: queries=1000, docs=1000, gold=1000
ne: queries=1000, docs=1000, gold=1000
or: queries=1000, docs=1000, gold=1000
pa: queries=1000, docs=1000, gold=1000
ta: queries=1000, docs=1000, gold=1000
te: queries=1000, docs=1000, gold=1000
ur: queries=1000, docs=1000, gold=1000


In [3]:
for lang in LANGS:

    files = {
        "BM25": RUN_DIR / "bm25" / f"{lang}_scores.npy",
        "BGE-M3": RUN_DIR / "bge-m3" / f"{lang}_scores.npy",
        "mE5-large": RUN_DIR / "mE5-large" / f"{lang}_scores.npy",
    }

    print(f"\n{lang}")

    for name, path in files.items():
        print(
            f"  {name:10s}: "
            f"{'OK' if path.exists() else 'MISSING'}"
        )


as
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

bn
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

gu
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

hi
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

kn
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

ml
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

mr
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

ne
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

or
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

pa
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

ta
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

te
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK

ur
  BM25      : OK
  BGE-M3    : MISSING
  mE5-large : OK


In [5]:
from pathlib import Path

ROOT = Path("..").resolve()

print("ROOT:")
print(ROOT)

print("\nRuns:")
runs = ROOT / "runs"
print(runs.exists())

if runs.exists():
    for p in runs.rglob("*"):
        if p.is_file():
            print(p.relative_to(ROOT))

print("\nResults:")
results = ROOT / "results"

for p in results.glob("*"):
    if p.is_file():
        print(p.name)

ROOT:
C:\Users\sanka\OneDrive\Desktop\research paper

Runs:
True
runs\bm25\as_ranks.npy
runs\bm25\as_scores.npy
runs\bm25\bn_ranks.npy
runs\bm25\bn_scores.npy
runs\bm25\gu_ranks.npy
runs\bm25\gu_scores.npy
runs\bm25\hi_ranks.npy
runs\bm25\hi_scores.npy
runs\bm25\kn_ranks.npy
runs\bm25\kn_scores.npy
runs\bm25\ml_ranks.npy
runs\bm25\ml_scores.npy
runs\bm25\mr_ranks.npy
runs\bm25\mr_scores.npy
runs\bm25\ne_ranks.npy
runs\bm25\ne_scores.npy
runs\bm25\or_ranks.npy
runs\bm25\or_scores.npy
runs\bm25\pa_ranks.npy
runs\bm25\pa_scores.npy
runs\bm25\ta_ranks.npy
runs\bm25\ta_scores.npy
runs\bm25\te_ranks.npy
runs\bm25\te_scores.npy
runs\bm25\ur_ranks.npy
runs\bm25\ur_scores.npy
runs\indicbert-v3\hi_ranks.npy
runs\indicbert-v3\hi_scores.npy
runs\mE5-base\as_ranks.npy
runs\mE5-base\as_scores.npy
runs\mE5-base\bn_ranks.npy
runs\mE5-base\bn_scores.npy
runs\mE5-base\gu_ranks.npy
runs\mE5-base\gu_scores.npy
runs\mE5-base\hi_ranks.npy
runs\mE5-base\hi_scores.npy
runs\mE5-base\kn_ranks.npy
runs\mE5-base\

In [6]:
import json

bge_file = Path("../results/bge_m3_hi.json")

print("Exists:", bge_file.exists())

if bge_file.exists():
    with open(bge_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Type:", type(data))

    if isinstance(data, dict):
        print("Keys:")
        print(data.keys())

    elif isinstance(data, list):
        print("Length:", len(data))
        print("First item:")
        print(data[0])

Exists: True
Type: <class 'dict'>
Keys:
dict_keys(['model', 'language', 'num_queries', 'corpus_size', 'mrr', 'recall@1', 'recall@5', 'recall@10', 'recall@20', 'recall@100', 'corpus_encoding_minutes', 'query_encoding_minutes', 'retrieval_seconds', 'total_minutes'])


In [7]:
from pathlib import Path

emb_dir = Path("../embeddings")

print("Embeddings directory exists:", emb_dir.exists())

if emb_dir.exists():
    files = list(emb_dir.rglob("*"))

    print("Number of files:", len(files))

    for f in files:
        if f.is_file():
            print(f.relative_to(emb_dir))

Embeddings directory exists: True
Number of files: 109
indicbert-v3_as_q.npy
indicbert-v3_hi_d.npy
indicbert-v3_hi_q.npy
indicbert_test.npy
mE5-base_as_d.npy
mE5-base_as_q.npy
mE5-base_bn_d.npy
mE5-base_bn_q.npy
mE5-base_gu_d.npy
mE5-base_gu_q.npy
mE5-base_hi_d.npy
mE5-base_hi_q.npy
mE5-base_kn_d.npy
mE5-base_kn_q.npy
mE5-base_ml_d.npy
mE5-base_ml_q.npy
mE5-base_mr_d.npy
mE5-base_mr_q.npy
mE5-base_ne_d.npy
mE5-base_ne_q.npy
mE5-base_or_d.npy
mE5-base_or_q.npy
mE5-base_pa_d.npy
mE5-base_pa_q.npy
mE5-base_ta_d.npy
mE5-base_ta_q.npy
mE5-base_te_d.npy
mE5-base_te_q.npy
mE5-base_ur_d.npy
mE5-base_ur_q.npy
mE5-large_as_d.npy
mE5-large_as_q.npy
mE5-large_bn_d.npy
mE5-large_bn_q.npy
mE5-large_gu_d.npy
mE5-large_gu_q.npy
mE5-large_hi_d.npy
mE5-large_hi_q.npy
mE5-large_kn_d.npy
mE5-large_kn_q.npy
mE5-large_ml_d.npy
mE5-large_ml_q.npy
mE5-large_mr_d.npy
mE5-large_mr_q.npy
mE5-large_ne_d.npy
mE5-large_ne_q.npy
mE5-large_or_d.npy
mE5-large_or_q.npy
mE5-large_pa_d.npy
mE5-large_pa_q.npy
mE5-large_ta